In [1]:
import datetime, warnings, scipy 
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import ConnectionPatch
from collections import OrderedDict
from matplotlib.gridspec import GridSpec
from mpl_toolkits.basemap import Basemap
from sklearn import metrics, linear_model
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from scipy.optimize import curve_fit
plt.rcParams["patch.force_edgecolor"] = True
plt.style.use('fivethirtyeight')
mpl.rc('patch', edgecolor = 'dimgray', linewidth=1)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "last_expr"
pd.options.display.max_columns = 50
%matplotlib inline
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('../flight-delay-data/flights.csv', low_memory=False)
df[:5]

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,2354.0,-11.0,21.0,15.0,205.0,194.0,169.0,1448,404.0,4.0,430,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,2.0,-8.0,12.0,14.0,280.0,279.0,263.0,2330,737.0,4.0,750,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,18.0,-2.0,16.0,34.0,286.0,293.0,266.0,2296,800.0,11.0,806,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,15.0,-5.0,15.0,30.0,285.0,281.0,258.0,2342,748.0,8.0,805,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,24.0,-1.0,11.0,35.0,235.0,215.0,199.0,1448,254.0,5.0,320,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
airports = pd.read_csv("../flight-delay-data/airports.csv")
airports[:2]

,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.4404
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.6819


In [4]:
df = df[df['MONTH'] == 1]
df['DATE'] = pd.to_datetime(df[['YEAR','MONTH', 'DAY']])

In [5]:
#_________________________________________________________
# Function that convert the 'HHMM' string to datetime.time
def format_heure(chaine):
    if pd.isnull(chaine):
        return np.nan
    else:
        if chaine == 2400: chaine = 0
        chaine = "{0:04d}".format(int(chaine))
        heure = datetime.time(int(chaine[0:2]), int(chaine[2:4]))
        return heure
#_____________________________________________________________________
# Function that combines a date and time to produce a datetime.datetime
def combine_date_heure(x):
    if pd.isnull(x[0]) or pd.isnull(x[1]):
        return np.nan
    else:
        return datetime.datetime.combine(x[0],x[1])
#_______________________________________________________________________________
# Function that combine two columns of the dataframe to create a datetime format
def create_flight_time(df, col):    
    liste = []
    for index, cols in df[['DATE', col]].iterrows():    
        if pd.isnull(cols[1]):
            liste.append(np.nan)
        elif float(cols[1]) == 2400:
            cols[0] += datetime.timedelta(days=1)
            cols[1] = datetime.time(0,0)
            liste.append(combine_date_heure(cols))
        else:
            cols[1] = format_heure(cols[1])
            liste.append(combine_date_heure(cols))
    return pd.Series(liste)

In [6]:
df['SCHEDULED_DEPARTURE'] = create_flight_time(df, 'SCHEDULED_DEPARTURE')
df['DEPARTURE_TIME'] = df['DEPARTURE_TIME'].apply(format_heure)
df['SCHEDULED_ARRIVAL'] = df['SCHEDULED_ARRIVAL'].apply(format_heure)
df['ARRIVAL_TIME'] = df['ARRIVAL_TIME'].apply(format_heure)
#__________________________________________________________________________
df.loc[:5, ['SCHEDULED_DEPARTURE', 'SCHEDULED_ARRIVAL', 'DEPARTURE_TIME',
             'ARRIVAL_TIME', 'DEPARTURE_DELAY', 'ARRIVAL_DELAY']]

,SCHEDULED_DEPARTURE,SCHEDULED_ARRIVAL,DEPARTURE_TIME,ARRIVAL_TIME,DEPARTURE_DELAY,ARRIVAL_DELAY
0,2015-01-01 00:05:00,04:30:00,23:54:00,04:08:00,-11.0,-22.0
1,2015-01-01 00:10:00,07:50:00,00:02:00,07:41:00,-8.0,-9.0
2,2015-01-01 00:20:00,08:06:00,00:18:00,08:11:00,-2.0,5.0
3,2015-01-01 00:20:00,08:05:00,00:15:00,07:56:00,-5.0,-9.0
4,2015-01-01 00:25:00,03:20:00,00:24:00,02:59:00,-1.0,-21.0
5,2015-01-01 00:25:00,06:02:00,00:20:00,06:10:00,-5.0,8.0


In [7]:
variables_to_remove = ['TAXI_OUT', 'TAXI_IN', 'WHEELS_ON', 'WHEELS_OFF', 'YEAR', 
                       'MONTH','DAY','DAY_OF_WEEK','DATE', 'AIR_SYSTEM_DELAY',
                       'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY',
                       'WEATHER_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON',
                       'FLIGHT_NUMBER', 'TAIL_NUMBER', 'AIR_TIME']
df.drop(variables_to_remove, axis = 1, inplace = True)
df = df[['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
        'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY',
        'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY',
        'SCHEDULED_TIME', 'ELAPSED_TIME']]
df[:5]

,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,SCHEDULED_TIME,ELAPSED_TIME
0,AS,ANC,SEA,2015-01-01 00:05:00,23:54:00,-11.0,04:30:00,04:08:00,-22.0,205.0,194.0
1,AA,LAX,PBI,2015-01-01 00:10:00,00:02:00,-8.0,07:50:00,07:41:00,-9.0,280.0,279.0
2,US,SFO,CLT,2015-01-01 00:20:00,00:18:00,-2.0,08:06:00,08:11:00,5.0,286.0,293.0
3,AA,LAX,MIA,2015-01-01 00:20:00,00:15:00,-5.0,08:05:00,07:56:00,-9.0,285.0,281.0
4,AS,SEA,ANC,2015-01-01 00:25:00,00:24:00,-1.0,03:20:00,02:59:00,-21.0,235.0,215.0


In [8]:
df.dropna(inplace = True)
print(df.shape[0])

457013


In [9]:
df.to_csv('/home/jovyan/shared/flight-delay-analysis/flight-delay-data/cleaned_flight_data.csv', index=False)

In [11]:
print(df.shape[0])

469968


In [14]:
from dataflow.dataflow import Dataflow
from sqlalchemy import text

dataflow = Dataflow()

db = dataflow.connection("supabase_dummy")

In [15]:
# Insert into delay_distribution
bins = list(range(-60, 301, 10))  # example: from -60 to 300 minutes, step = 10
df['delay_bin'] = pd.cut(df['DEPARTURE_DELAY'], bins=bins, right=False)

# Count per bin
hist = df['delay_bin'].value_counts().sort_index()

# Prepare data
rows = []
for interval, count in hist.items():
    rows.append({
        "bin_start": int(interval.left),
        "bin_end": int(interval.right),
        "count": int(count)
    })

# Insert
from sqlalchemy import text
with db.begin():
    for row in rows:
        db.execute(
            text("""
                INSERT INTO delay_distribution (bin_start, bin_end, count)
                VALUES (:bin_start, :bin_end, :count)
            """),
            row
        )

In [16]:
# Insert into flight_statistics_summary

summary_df = df.groupby('AIRLINE').agg(
    total_flights=('AIRLINE', 'count'),
    avg_departure_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arrival_delay=('ARRIVAL_DELAY', 'mean')
).reset_index()

with db.begin():
    for _, row in summary_df.iterrows():
        db.execute(
            text("""
                INSERT INTO flight_statistics_summary (
                    airline, total_flights, avg_departure_delay, avg_arrival_delay
                ) VALUES (
                    :airline, :total_flights, :avg_departure_delay, :avg_arrival_delay
                )
            """),
            {
                "airline": row['AIRLINE'],
                "total_flights": int(row['total_flights']),
                "avg_departure_delay": float(row['avg_departure_delay']),
                "avg_arrival_delay": float(row['avg_arrival_delay']),
            }
        )

In [ ]:
# Insert into delay_vs_hour

df['hour'] = pd.to_datetime(df['SCHEDULED_DEPARTURE']).dt.hour

hourly_df = df.groupby(['hour', 'AIRLINE']).agg(
    avg_departure_delay=('DEPARTURE_DELAY', 'mean')
).reset_index()

with db.begin():
    for _, row in hourly_df.iterrows():
        db.execute(
            text("""
                INSERT INTO delay_vs_hour (
                    hour, airline, avg_departure_delay
                ) VALUES (
                    :hour, :airline, :avg_departure_delay
                )
            """),
            {
                "hour": int(row['hour']),
                "airline": row['AIRLINE'],
                "avg_departure_delay": float(row['avg_departure_delay']),
            }
        )

In [32]:
# hourly_avg_delay

hourly_df = df.copy()
hourly_df['hour'] = pd.to_datetime(hourly_df['SCHEDULED_DEPARTURE']).dt.hour

agg_hourly = (
    hourly_df.groupby('hour')['DEPARTURE_DELAY']
    .mean()
    .reset_index(name='avg_departure_delay')
)

with db.begin():
    for _, row in agg_hourly.iterrows():
        db.execute(text("""
            INSERT INTO hourly_avg_delay (hour, avg_departure_delay)
            VALUES (:hour, :avg)
        """), {"hour": int(row['hour']), "avg": float(row['avg_departure_delay'])})

In [33]:
# top10_origin_airports_by_delay

agg_origin = (
    df.groupby('ORIGIN_AIRPORT')['DEPARTURE_DELAY']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='avg_departure_delay')
)

with db.begin():
    for _, row in agg_origin.iterrows():
        db.execute(text("""
            INSERT INTO top10_origin_airports_by_delay (origin_airport, avg_departure_delay)
            VALUES (:airport, :avg)
        """), {"airport": row['ORIGIN_AIRPORT'], "avg": float(row['avg_departure_delay'])})


In [34]:
# weekday_avg_delay

weekday_df = df.copy()
weekday_df['day_of_week'] = pd.to_datetime(weekday_df['SCHEDULED_DEPARTURE']).dt.day_name()

agg_weekday = (
    weekday_df.groupby('day_of_week')[['DEPARTURE_DELAY', 'ARRIVAL_DELAY']]
    .mean()
    .reset_index()
    .rename(columns={
        'DEPARTURE_DELAY': 'avg_departure_delay',
        'ARRIVAL_DELAY': 'avg_arrival_delay'
    })
)

with db.begin():
    for _, row in agg_weekday.iterrows():
        db.execute(text("""
            INSERT INTO weekday_avg_delay (day_of_week, avg_departure_delay, avg_arrival_delay)
            VALUES (:dow, :dep, :arr)
        """), {
            "dow": row['day_of_week'],
            "dep": float(row['avg_departure_delay']),
            "arr": float(row['avg_arrival_delay']),
        })

In [35]:
# origin_airport_stats

origin_stats = (
    df.groupby('ORIGIN_AIRPORT')
    .agg(
        flight_count=('ORIGIN_AIRPORT', 'count'),
        avg_departure_delay=('DEPARTURE_DELAY', 'mean')
    )
    .reset_index()
)

with db.begin():  # This starts a transaction and commits automatically at the end
    for _, row in origin_stats.iterrows():
        db.execute(text("""
            INSERT INTO origin_airport_stats (origin_airport, flight_count, avg_departure_delay)
            VALUES (:origin, :count, :delay)
        """), {
            "origin": row['ORIGIN_AIRPORT'],
            "count": int(row['flight_count']),
            "delay": float(row['avg_departure_delay']),
        })

In [36]:
# Insert into airport_details

with db.begin():
    for _, row in airports.iterrows():
        db.execute(text("""
            INSERT INTO airport_details (iata_code, airport, city, state, latitude, longitude)
            VALUES (:code, :airport, :city, :state, :lat, :lon)
        """), {
            "code": row['IATA_CODE'],
            "airport": row['AIRPORT'],
            "city": row['CITY'],
            "state": row['STATE'],
            "lat": float(row['LATITUDE']),
            "lon": float(row['LONGITUDE']),
        })

In [37]:
# Preprocess airline performance data
airline_stats = df.groupby('AIRLINE').agg(
    avg_departure_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arrival_delay=('ARRIVAL_DELAY', 'mean'),
    total_flights=('AIRLINE', 'count'),
    on_time_pct=('ARRIVAL_DELAY', lambda x: (x <= 0).sum() / len(x) * 100)
).reset_index()

with db.begin():
    for _, row in airline_stats.iterrows():
        db.execute(text("""
            INSERT INTO airline_performance_stats (
                airline, avg_departure_delay, avg_arrival_delay,
                total_flights, on_time_pct
            ) VALUES (
                :airline, :avg_dep, :avg_arr, :total, :on_time
            )
        """), {
            "airline": row['AIRLINE'],
            "avg_dep": float(row['avg_departure_delay']),
            "avg_arr": float(row['avg_arrival_delay']),
            "total": int(row['total_flights']),
            "on_time": float(row['on_time_pct'])
        })
